* 경로 설정

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day01"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = SANDBOX / "samples"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


* 표 추출

In [2]:
from docx import Document

DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))

table = Document(str(DOCX)).tables[0]
flat = [cell.text.strip() for row in table.rows for cell in row.cells]

print(f'표 크기 : {len(table.rows)}행 {len(table.columns)}열')
print()
print(' '.join(flat))

표 크기 : 4행 7열

버전 구분 시행일 주요 내용 작성 검토 승인 v2.0 개정 2025-07-01 일비 인상(25,000→30,000) · 광역시 숙박비 상한 신설 · 별표1 전면 개정 김지원 박성호 대표이사 v1.1 개정 2024-09-01 교통비 정산 방식을 실비 정산으로 변경 김지원 박성호 대표이사 v1.0 제정 2024-01-01 최초 제정 이한별 박성호 대표이사


* 마크다운으로 변환해주는 코드

In [3]:
"""표의 머리글과 나머지 행들을 마크다운 표 한 덩어리로 바꾼다."""
def _table_to_markdown(headers: list[str], rows: list[list[str]]) -> str: 
    head = '| ' + ' | '.join(headers) + ' |'
    sep = '|' + '---|' * len(headers)
    body = ['| ' + ' | '.join((str(c) for c in r)) + ' |' for r in rows]
    return '\n'.join([head, sep, *body])
    
print(_table_to_markdown(['버전', '구분', '시행일'], [['v2.0', '개정', '2025-07-01']]))

| 버전 | 구분 | 시행일 |
|---|---|---|
| v2.0 | 개정 | 2025-07-01 |


In [4]:
rows = [[cell.text.strip() for cell in row.cells] for row in table.rows]
print(_table_to_markdown(rows[0], rows[1:]))
print()
print('첫 행을 머리로 씁니다 - 머리 행이 없는 표도 마찬가지입니다.')

| 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|---|---|---|
| v2.0 | 개정 | 2025-07-01 | 일비 인상(25,000→30,000) · 광역시 숙박비 상한 신설 · 별표1 전면 개정 | 김지원 | 박성호 | 대표이사 |
| v1.1 | 개정 | 2024-09-01 | 교통비 정산 방식을 실비 정산으로 변경 | 김지원 | 박성호 | 대표이사 |
| v1.0 | 제정 | 2024-01-01 | 최초 제정 | 이한별 | 박성호 | 대표이사 |

첫 행을 머리로 씁니다 - 머리 행이 없는 표도 마찬가지입니다.


In [5]:
import importlib
import app.rag.local_parsers as lp
importlib.reload(lp)         # 파일 다시 읽어오기     
parse_docx = lp.parse_docx

In [6]:
from app.rag.local_parsers import parse_docx

doc = parse_docx(DOCX)
print(f"블록 {len(doc.blocks)} / table_count {doc.table_count}")
print()
print("첫 표 블럭의 내용 : ")
first = next(b for b in doc.blocks if b.kind == "표") 
# print(first.text)
# print(first.text.splitlines()) # 하나의 문자열을 줄단위로 분리해 리스트로 받기 
print("\n".join(first.text.splitlines()))

블록 233 / table_count 7

첫 표 블럭의 내용 : 
| 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|---|---|---|
| v2.0 | 개정 | 2025-07-01 | 일비 인상(25,000→30,000) · 광역시 숙박비 상한 신설 · 별표1 전면 개정 | 김지원 | 박성호 | 대표이사 |
| v1.1 | 개정 | 2024-09-01 | 교통비 정산 방식을 실비 정산으로 변경 | 김지원 | 박성호 | 대표이사 |
| v1.0 | 제정 | 2024-01-01 | 최초 제정 | 이한별 | 박성호 | 대표이사 |


In [7]:
from pypdf import PdfReader

PDF = next(SAMPLES.glob('DOC-HR-014_*_v2.0.pdf'))

reader = PdfReader(str(PDF))
lines = (reader.pages[1].extract_text() or '').splitlines()

for n, line in enumerate(lines[:25], 1):
    print(f'{n:2d} | {line}')
    
print('...')
total = sum((len((p.extract_text() or '').splitlines()) for p in reader.pages))

print(f'(문서 전체 {total}줄)')

 1 | 제·개정 이력
 2 | 버전
 3 | 구분
 4 | 시행일
 5 | 주요 내용
 6 | 작성
 7 | 검토
 8 | 승인
 9 | v2.0
10 | 개정
11 | 2025-07-01
12 | 일비
13 | 인상(25,000→30,000)
14 | · 광역시 숙박비 상한
15 | 신설 · 별표1 전면 개정
16 | 김지원
17 | 박성호
18 | 대표이사
19 | v1.1
20 | 개정
21 | 2024-09-01
22 | 교통비 정산 방식을 실비
23 | 정산으로 변경
24 | 김지원
25 | 박성호
...
(문서 전체 434줄)


In [8]:
import pdfplumber

with pdfplumber.open(str(PDF)) as pdf:
    tables = pdf.pages[1].extract_tables()
    
t = tables[0]

print(f'이 쪽의 표 : {len(tables)}개')
print(f'{len(t)}행 {len(t[0])}열')
print(t[0])
print(t[1])


이 쪽의 표 : 1개
4행 7열
['버전', '구분', '시행일', '주요 내용', '작성', '검토', '승인']
['v2.0', '개정', '2025-07-01', '일비\n인상(25,000→30,000)\n· 광역시 숙박비 상한\n신설 · 별표1 전면 개정', '김지원', '박성호', '대표이사']


In [9]:
found = 0
with pdfplumber.open(str(PDF)) as pdf:
    for n, page in enumerate(pdf.pages, 1):
        k = len(page.extract_tables())
        if k:
            print(f'p.{n:2d} : {k}개')
            found += k
print('─' * 13)
print(f'PDF 표 합계 : {found}   ·   DOCX 표 : {doc.table_count}   ·   같다 : {found == doc.table_count}')

p. 2 : 1개
p.11 : 1개
p.12 : 1개
p.13 : 1개
p.14 : 1개
p.15 : 1개
p.16 : 1개
─────────────
PDF 표 합계 : 7   ·   DOCX 표 : 7   ·   같다 : True


In [10]:

print(_table_to_markdown(t[0], t[1:]))
print()

with pdfplumber.open(str(PDF)) as pdf:
    cells = [c for page in pdf.pages for tb in page.extract_tables() for row in tb for c in row]
nl = sum((1 for c in cells if c and '\n' in c))

print(f'줄바꿈이 든 셀 : {nl}개 (전체 {len(cells)}개 중)') # 표가 깨져서 출력 

| 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|---|---|---|
| v2.0 | 개정 | 2025-07-01 | 일비
인상(25,000→30,000)
· 광역시 숙박비 상한
신설 · 별표1 전면 개정 | 김지원 | 박성호 | 대표이사 |
| v1.1 | 개정 | 2024-09-01 | 교통비 정산 방식을 실비
정산으로 변경 | 김지원 | 박성호 | 대표이사 |
| v1.0 | 제정 | 2024-01-01 | 최초 제정 | 이한별 | 박성호 | 대표이사 |

줄바꿈이 든 셀 : 8개 (전체 220개 중)


In [11]:
# 셀 안의 줄바꿈 \n 을 띄어쓰기로 변경 
flat = [[(c or "").replace("\n", " ").strip() for c in row] for row in t]

print(_table_to_markdown(flat[0], flat[1:]))

| 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|---|---|---|
| v2.0 | 개정 | 2025-07-01 | 일비 인상(25,000→30,000) · 광역시 숙박비 상한 신설 · 별표1 전면 개정 | 김지원 | 박성호 | 대표이사 |
| v1.1 | 개정 | 2024-09-01 | 교통비 정산 방식을 실비 정산으로 변경 | 김지원 | 박성호 | 대표이사 |
| v1.0 | 제정 | 2024-01-01 | 최초 제정 | 이한별 | 박성호 | 대표이사 |


* 표를 파일로 저장 (추후 다른 포맷과 비교하기위한 준비)

In [12]:
parts = [f'## {b.locator}\n{b.text}\n' for b in doc.blocks if b.kind == '표']
md_text = '\n'.join(parts)

out = SANDBOX / 'tables_DOC-HR-014_v2.0.md'
out.write_text(md_text, encoding='utf-8')

print(f'저장 : {out.relative_to(ROOT).as_posix()}')
print(f'크기 : {out.stat().st_size:,} 바이트 · {len(md_text.splitlines())}줄')
print()
print('다시 읽기 :')
print('\n'.join(out.read_text(encoding='utf-8').splitlines()[:2]))


저장 : sandbox/w5/day01/tables_DOC-HR-014_v2.0.md
크기 : 3,542 바이트 · 88줄

다시 읽기 :
## 표1
| 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |


In [13]:
try:
    from app.rag.local_parsers import parse_docx, parse_pdf
except ModuleNotFoundError:
    print('import 실패 — 맨 위 bootstrap 두 셀을 먼저 실행하세요.')
    print('그래도 안 되면 커널을 재시작(Kernel ▸ Restart)한 뒤 처음부터 다시 실행하세요.')
    raise
    
docx_path = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
pdf_path = next(SAMPLES.glob('DOC-HR-014_*_v2.0.pdf'))
docx_doc = parse_docx(docx_path)
pdf_doc = parse_pdf(pdf_path)
print('docx 블록', len(docx_doc.blocks))
print('pdf  블록 ', len(pdf_doc.blocks))

docx 블록 233
pdf  블록  16


In [14]:
print('형식        블록    표   page_count    글자(약)')
print('─' * 47)
for name, doc in (('docx', docx_doc), ('pdf', pdf_doc)):
    chars = sum((len(b.text) for b in doc.blocks))
    print(f'{name:<8}{len(doc.blocks):>8}{doc.table_count:>6}{doc.page_count:>13}{round(chars, -2):>10,}자')

형식        블록    표   page_count    글자(약)
───────────────────────────────────────────────
docx         233     7            1     8,300자
pdf           16     0           16     7,800자


In [15]:
for name, doc in (('docx', docx_doc), ('pdf', pdf_doc)):
    sizes = [len(b.text) for b in doc.blocks]
    if not sizes:
        print(f'{name:<5}: 블록 0개 — 잴 것이 없습니다')
        continue
    avg = round(sum(sizes) / len(sizes))
    print(f'{name:<5}: 블록 {len(sizes):>3}개 · 평균 약 {round(avg / 5) * 5}자 · 가장 짧은 블록 {min(sizes)}자 · 가장 긴 블록 {max(sizes)}자')


docx : 블록 233개 · 평균 약 35자 · 가장 짧은 블록 2자 · 가장 긴 블록 378자
pdf  : 블록  16개 · 평균 약 490자 · 가장 짧은 블록 108자 · 가장 긴 블록 1070자


In [16]:
path = ROOT / "backend" / "tests" / "test_local_parsers.py"
text = path.read_text(encoding="utf-8")
print(path.name, "·", len(text.splitlines()), "줄 · 테스트 함수", text.count("\ndef test_"), "개")

test_local_parsers.py · 51 줄 · 테스트 함수 3 개
